In [ ]:
from sentence_transformers import SentenceTransformer

# Модель скачается автоматически при инициализации
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)


/home/mivi/Documents/RT_second/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3862.61it/s]


(2, 384)


In [2]:
import ast
import json
from pathlib import Path

# Конфигурация путей (настроена под твой docker-compose)
REPO_ROOT = Path("./dataset_case3_v1.0_fix/gymhero")
OUTPUT_JSON = Path("./chroma_ready_chunks.json")


def get_node_source(lines: list[str], node: ast.AST) -> str:
    """Извлекает чистый исходный текст фрагмента кода из строк файла."""
    start_line = node.lineno - 1
    end_line = getattr(node, "end_lineno", len(lines))
    return "\n".join(lines[start_line:end_line])


def extract_chunks_from_file(py_file: Path, repo_root: Path) -> list[dict]:
    """Режет Python-файл на классы, методы и функции с помощью AST-парсера.

    Формирует структуру, полностью готовую для импорта в ChromaDB.
    """
    # Получаем относительный путь с прямыми слэшами (требование ТЗ)
    rel_path = py_file.relative_to(repo_root).as_posix()

    # Гарантируем наличие префикса gymhero/ в путях для точного совпадения со score.py
    if not rel_path.startswith("gymhero/"):
        rel_path = f"gymhero/{rel_path}"

    try:
        src = py_file.read_text(encoding="utf-8", errors="replace")
        tree = ast.parse(src)
    except (SyntaxError, Exception) as e:
        print(f"Ошибка парсинга файла {py_file}: {e}")
        return []

    lines = src.splitlines()
    chunks = []

    for node in tree.body:
        # 1. Обработка классов верхнего уровня
        if isinstance(node, ast.ClassDef):
            class_id = f"{rel_path}:{node.name}:{node.lineno}"
            chunks.append(
                {
                    "id": class_id,  # Пойдет в ids для ChromaDB
                    "document": get_node_source(
                        lines, node
                    ),  # Текст для генерации эмбеддинга
                    "metadata": {
                        "chunk_id": class_id,  # Дублируем для совместимости со score.py
                        "file_path": rel_path,
                        "type": "class",
                        "name": node.name,
                        "start_line": node.lineno,
                    },
                }
            )

            # Внутри класса ищем методы
            for sub_node in node.body:
                if isinstance(sub_node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                    method_name = f"{node.name}.{sub_node.name}"
                    method_id = f"{rel_path}:{method_name}:{sub_node.lineno}"
                    chunks.append(
                        {
                            "id": method_id,
                            "document": get_node_source(lines, sub_node),
                            "metadata": {
                                "chunk_id": method_id,
                                "file_path": rel_path,
                                "type": "method",
                                "name": method_name,
                                "start_line": sub_node.lineno,
                            },
                        }
                    )

        # 2. Обработка функций верхнего уровня (вне классов)
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            func_id = f"{rel_path}:{node.name}:{node.lineno}"
            chunks.append(
                {
                    "id": func_id,
                    "document": get_node_source(lines, node),
                    "metadata": {
                        "chunk_id": func_id,
                        "file_path": rel_path,
                        "type": "function",
                        "name": node.name,
                        "start_line": node.lineno,
                    },
                }
            )

    return chunks


def main():
    if not REPO_ROOT.exists():
        print(
            f"Ошибка: Директория {REPO_ROOT} не найдена. Проверьте конфигурацию путей."
        )
        return

    print(f"Начало обхода и парсинга репозитория: {REPO_ROOT}")

    all_chunks = []
    file_count = 0

    # Рекурсивный поиск всех файлов .py в gymhero
    for py_file in REPO_ROOT.rglob("*.py"):
        file_count += 1
        file_chunks = extract_chunks_from_file(py_file, REPO_ROOT.parent)
        all_chunks.extend(file_chunks)

    print(f"Успешно обработано файлов: {file_count}")
    print(f"Всего извлечено и подготовлено чанков: {len(all_chunks)}")

    # Сохраняем в единую структуру, которую легко распарсить одной строкой
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(all_chunks, f, ensure_ascii=False, indent=2)

    print(f"Данные успешно сохранены в {OUTPUT_JSON}")


if __name__ == "__main__":
    main()

Начало обхода и парсинга репозитория: dataset_case3_v1.0_fix/gymhero
Успешно обработано файлов: 47
Всего извлечено и подготовлено чанков: 155
Данные успешно сохранены в chroma_ready_chunks.json


In [ ]:
import json
import sys
from pathlib import Path
import chromadb
from chromadb.utils import embedding_functions



CHROMA_PATH = Path("./chroma_db")
COLLECTION_NAME = "code_chunks"
CHROMA_READY_JSON = Path("./chroma_ready_chunks.json")

# Модель из ТЗ
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"


def main():
    if not CHROMA_READY_JSON.exists():
        sys.exit(1)

    try:
        with open(CHROMA_READY_JSON, "r", encoding="utf-8") as f:
            chunks = json.load(f)
    except Exception:
        sys.exit(1)

    if not chunks:
        sys.exit(0)

    # Инициализируем функцию эмбеддингов
    sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name=MODEL_NAME
    )

    client = chromadb.PersistentClient(path=str(CHROMA_PATH))

    try:
        client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass

    collection = client.create_collection(
        name=COLLECTION_NAME, embedding_function=sentence_transformer_ef
    )

    # Подготовка списков для пакетной вставки
    ids = [item["id"] for item in chunks]
    documents = [item["document"] for item in chunks]
    metadatas = [item["metadata"] for item in chunks]

    # Пакетная тихая запись
    batch_size = 400
    total_chunks = len(documents)

    for i in range(0, total_chunks, batch_size):
        end_idx = min(i + batch_size, total_chunks)

        collection.add(
            ids=ids[i:end_idx],
            documents=documents[i:end_idx],
            metadatas=metadatas[i:end_idx],
        )

    # sys.exit(0)


if __name__ == "__main__":
    main()